In [1]:
import numpy as np
import pandas as pd
import anndata as ad


In [7]:
expr_df = pd.read_csv("/cluster/work/boeva/eheiss/datasets/GDSC/drug_response_expr_data.csv", index_col=0, low_memory=False)
expr_df.index = expr_df.index.astype(str)
print(expr_df.shape)
expr_df.head()


(700, 19104)


,cell_line_display_name,lineage_1,lineage_2,lineage_3,lineage_6,lineage_4,NEMP2,SPDYE11,MED6,SPATA1,...,XYLB,CDC25A,NR1H4,NUP153,SUPT7L,GFPT2,USP15,IQSEC1,FGFBP1,FGF19
depmap_id,,,,,,,,,,,,,,,,,,,,,
ACH-000973,639V,Bladder/Urinary Tract,Urethral Cancer,Urethral Urothelial Carcinoma,NaN,NaN,2.350843,0.028249,5.144697,0.501200,...,2.911525,4.919169,-0.004244,5.058834,4.850728,3.627488,5.157255,3.509118,0.619068,-0.012049
ACH-000757,A427,Lung,Non-Small Cell Lung Cancer,Lung Adenocarcinoma,NaN,NaN,2.291779,-0.007359,5.119924,1.057133,...,2.174921,3.924597,-0.004244,4.806072,4.504631,2.145680,5.489351,2.646765,0.068688,0.014902
ACH-000248,AU565,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,HER2+,NaN,0.665334,-0.007359,5.392927,0.767300,...,2.852315,3.967462,0.036676,5.313185,4.651704,0.690699,5.182693,4.079506,0.205939,-0.012049
ACH-001016,BECKER,CNS/Brain,Diffuse Glioma,Astrocytoma,NaN,NaN,1.581257,0.016474,5.572271,0.146312,...,1.085777,3.120451,0.165230,4.544086,5.531915,4.626524,5.355187,2.052331,0.291948,1.389632
ACH-000245,BL41,Lymphoid,Mature B-Cell Neoplasms,Burkitt Lymphoma,NaN,NaN,2.194331,-0.007359,5.231515,0.482211,...,2.311855,5.645543,-0.004244,5.460451,4.136652,0.102620,5.233888,3.945063,0.183967,0.054353


In [8]:
gene_info = pd.read_csv("/cluster/work/boeva/eheiss/scbFM/data/bulkformer_gene_info.csv")
sym2ensg = dict(zip(gene_info["gene_symbol"].astype(str), gene_info["ensg_id"].astype(str)))

symbol_cols = [c for c in expr_df.columns if c in sym2ensg]
ensg_ids = [sym2ensg[c] for c in symbol_cols]

expr_mapped = expr_df[symbol_cols].copy()
expr_mapped.columns = ensg_ids
expr_mapped = expr_mapped.loc[:, ~expr_mapped.columns.duplicated(keep="first")]

n_mapped = len(expr_mapped.columns)
n_unmapped = len(expr_df.columns) - len(symbol_cols)
print(f"Mapped: {n_mapped} / {len(expr_df.columns)}  |  Unmapped: {n_unmapped}")


Mapped: 18916 / 19104  |  Unmapped: 188


In [ ]:
adata = ad.AnnData(X=expr_mapped.values.astype("float32"))
adata.obs_names = list(expr_mapped.index)
adata.var_names = list(expr_mapped.columns)
adata

In [10]:
ic50 = pd.read_csv("/cluster/work/boeva/eheiss/datasets/GDSC/drug_response_prediction_IC50.csv")
ic50_cell_ids = set(ic50["ModelID"].astype(str))
expr_cell_ids = set(adata.obs_names)
overlap = ic50_cell_ids & expr_cell_ids
print(f"Cell lines in IC50: {len(ic50_cell_ids)}")
print(f"Cell lines in expression: {len(expr_cell_ids)}")
print(f"Overlap: {len(overlap)}")


Cell lines in IC50: 700
Cell lines in expression: 700
Overlap: 700


In [11]:
adata.write("/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad")
